In [4]:
# 외부의 텍스트 파일을 로드하는 방법
text = open('../data/qa.txt', 'r', encoding='utf-8').read()

In [7]:
text

'[\n    ("환불은 어떻게 하나요?", "주문 상세 페이지에서 \'환불 신청\' 버튼을 눌러 접수하실 수 있습니다."),\n    ("배송 기간은 얼마나 걸리나요?", "일반 배송은 2~3일, 도서산간 지역은 최대 5일까지 소요됩니다."),\n    ("해외 배송도 가능한가요?", "현재 해외 배송은 지원하지 않습니다."),\n    ("회원 탈퇴는 어디에서 하나요?", "설정 > 계정 관리 > 회원 탈퇴 메뉴에서 진행하실 수 있습니다."),\n    ("비밀번호를 잊어버렸어요", "로그인 화면의 \'비밀번호 재설정\' 링크를 통해 재설정 가능합니다."),\n    ("주문 취소는 어떻게 하죠?", "상품이 배송 준비 전 상태라면 주문 상세 페이지에서 취소가 가능합니다."),\n    ("영수증 발급이 가능한가요?", "마이페이지 > 주문 내역에서 영수증 출력이 가능합니다."),\n    ("교환/반품은 가능한가요?", "수령일로부터 7일 이내, 미사용/미훼손 제품에 한해 가능합니다."),\n]'

In [8]:
# 일반 문자를 python의 구조로 변경하는 함수
QnA_list = eval(text)

In [10]:
type(QnA_list)

list

- QnA_list 데이터에서 질문들을 따로 추출하여 형태소 분석을 이용하여 단어를 추출
- 벡터화 작업 (TF-IDF)
- 코사인 유사도
    - 문장과 문장 사이에 어느 정도 같은 의미를 가지는가?

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
# QnA_list에서 질문들의 목록을 생성
# 패턴 인지 -> list에서 각각의 원소들을 추출 -> 원소에서 첫번쨰 문자를 추출
# 목록이라는 새로운 리스트를 생성
questions = []
# list에서 각각의 원소들을 추출
for data in QnA_list:
    print(data[0])

환불은 어떻게 하나요?
배송 기간은 얼마나 걸리나요?
해외 배송도 가능한가요?
회원 탈퇴는 어디에서 하나요?
비밀번호를 잊어버렸어요
주문 취소는 어떻게 하죠?
영수증 발급이 가능한가요?
교환/반품은 가능한가요?


In [26]:
for q, a in QnA_list:
    print(q)

환불은 어떻게 하나요?
배송 기간은 얼마나 걸리나요?
해외 배송도 가능한가요?
회원 탈퇴는 어디에서 하나요?
비밀번호를 잊어버렸어요
주문 취소는 어떻게 하죠?
영수증 발급이 가능한가요?
교환/반품은 가능한가요?


In [28]:
questions = [q for q, a in QnA_list]
questions

['환불은 어떻게 하나요?',
 '배송 기간은 얼마나 걸리나요?',
 '해외 배송도 가능한가요?',
 '회원 탈퇴는 어디에서 하나요?',
 '비밀번호를 잊어버렸어요',
 '주문 취소는 어떻게 하죠?',
 '영수증 발급이 가능한가요?',
 '교환/반품은 가능한가요?']

In [29]:
from konlpy.tag import Komoran

In [30]:
# 토큰화 -> 벡터화
komoran = Komoran()

def tokenize(text):
    return komoran.morphs(text)

vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    ngram_range=(1, 2),
    lowercase=False
)

In [31]:
X = vectorizer.fit_transform(questions)

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [33]:
X.toarray().shape

(8, 76)

In [36]:
# 새로운 질문
query = "환불을 하려면 어떻게 하면 될까요?"
# 질문을 벡터화
query_vec = vectorizer.transform([query])

In [41]:
# 코사인 거리 유사도 함수를 사용
# ravel() -> array에서 사용하는 함수로 다차원 배열을 1차원 배열로 변경하는 함수
sims = cosine_similarity(query_vec, X).ravel()
sims

array([0.6897874 , 0.02521506, 0.10545181, 0.0905098 , 0.        ,
       0.45330041, 0.10370606, 0.09595719])

In [43]:
# argsort() : 배열의 값을 정렬했을때 그 정렬 순서를 되돌려주는 함수
# 정렬의 순서는 인덱스의 의미
rank = sims.argsort()[::-1]

In [49]:
rank

array([0, 5, 2, 6, 7, 3, 1, 4])

In [ ]:
print('질문 :', query)
for i in rank[:2]:
    print(f"index : {i},  유사도 :{round(sims[i], 3)},  유사 질문 : {questions[i]},  답변 : {QnA_list[i][1]}")

질문 : 환불을 하려면 어떻게 하면 될까요?
0
index : 0,  유사도 :0.69,  유사 질문 : 환불은 어떻게 하나요?,  답변 : 주문 상세 페이지에서 '환불 신청' 버튼을 눌러 접수하실 수 있습니다.
5
index : 5,  유사도 :0.453,  유사 질문 : 주문 취소는 어떻게 하죠?,  답변 : 상품이 배송 준비 전 상태라면 주문 상세 페이지에서 취소가 가능합니다.
